# omcwa fixed-pipeline showcase

This notebook uses the committed synthetic fixtures, so it runs from a clean
checkout without local recordings or external binaries. `process_cwa` follows
one native path: load once, optionally auto-calibrate, then resample.
Calibration failures are strict by default.

Install the showcase dependencies before running:

```bash
uv sync --group showcase
```

In [ ]:
from pathlib import Path

import numpy as np

from omcwa import CalibrationError, InterpolateMode, load_cwa, process_cwa

# Works whether Jupyter starts in the repository root or examples/.
FIXTURE_DIR = Path("tests/fixtures/golden")
if not FIXTURE_DIR.is_dir():
    FIXTURE_DIR = Path("../tests/fixtures/golden")

CAL_SUCCESS = FIXTURE_DIR / "cal_success.cwa"
CAL_FAILURE = FIXTURE_DIR / "cal_failure.cwa"
CAL_FAILURE_NO_AXIS = FIXTURE_DIR / "cal_failure_no_axis.cwa"

for fixture in (CAL_SUCCESS, CAL_FAILURE, CAL_FAILURE_NO_AXIS):
    if not fixture.is_file():
        raise FileNotFoundError(f"Fixture not found: {fixture}")

print(f"Using fixtures from: {FIXTURE_DIR}")

## Default strict pipeline

The default uses cubic interpolation at the file rate. The successful fixture
contains enough stationary orientations for auto-calibration to converge.

In [ ]:
default_out = process_cwa(CAL_SUCCESS)

print(f"sample rate: {default_out.sample_rate_hz} Hz")
print(f"acc shape: {default_out.acc.shape} (g)")
print(f"gyr shape: {default_out.gyr.shape} (dps)")
print(f"calibration success: {default_out.calibration.success}")
print(f"calibration error code: {default_out.calibration.error_code}")
print(f"metadata keys: {sorted(default_out.metadata)}")

## Strict calibration failures

Failed fits raise `CalibrationError` before resampled output is allocated. The
exception preserves omconvert's native error code.

In [ ]:
for fixture in (CAL_FAILURE, CAL_FAILURE_NO_AXIS):
    try:
        process_cwa(fixture)
    except CalibrationError as error:
        print(f"{fixture.name}: error code {error.error_code}")

## Explicit identity fallback and no-calibration mode

`on_calibration_failure="identity"` accepts omconvert's identity coefficients
but keeps `success=False` and the failure code. `calibrate=False` skips fitting
and reports a successful identity calibration.

In [ ]:
fallback = process_cwa(
    CAL_FAILURE,
    on_calibration_failure="identity",
)
no_cal = process_cwa(CAL_FAILURE, calibrate=False)

print(
    "fallback: ",
    fallback.calibration.success,
    fallback.calibration.error_code,
)
print(
    "no calibration: ",
    no_cal.calibration.success,
    no_cal.calibration.error_code,
)
print(f"max accel difference: {np.max(np.abs(fallback.acc - no_cal.acc))}")

## Uncalibrated loading, explicit rate, and output windows

`load_cwa` exposes file-rate acceleration, gyroscope, and temperature with
identity calibration. `process_cwa` accepts a target rate, interpolation mode,
and a half-open Unix-time output range.

In [ ]:
uniform = load_cwa(CAL_SUCCESS)
t0 = float(uniform.time[0])

windowed = process_cwa(
    CAL_SUCCESS,
    sample_rate_hz=50.0,
    interpolate=InterpolateMode.LINEAR,
    time_range=(t0 + 1.0, t0 + 3.0),
)

print(f"temperature shape: {uniform.temp.shape} (degrees C)")
print(f"windowed shape: {windowed.acc.shape}")
print(f"windowed rate: {windowed.sample_rate_hz} Hz")
print(f"time bounds: {windowed.time[0]:.2f} .. {windowed.time[-1]:.2f}")

## Current limits and future extension work

The complete first session and output are materialized. `time_range` trims only
after processing, and version 0.1 makes no 1 GB deployment-readiness claim.
Custom backends remain a non-public TODO until a lazy, pre-resample,
chunk-capable `CwaSource` contract is designed.

In [ ]:
assert all(not key.startswith("_") for key in default_out.metadata)

print(f"valid samples: {np.count_nonzero(default_out.valid)}")
print(f"clipped samples: {np.count_nonzero(default_out.clipped)}")
print("No native handles or private metadata keys are retained.")

## Production use

Keep strict calibration failures unless identity fallback is an explicit data
quality policy. For large recordings, measure peak memory in the target
environment and wait for the planned native windowing/chunking work before
making deployment-scale guarantees.